# 07h — Train & Evaluate (GIN(E) vs GATv2 Comparison, Pooled Bogor+Warsaw)

Seventh parallel branch. Same pooled Bogor+Warsaw data, same plain
random 70/15/15 split, same capacity numbers (`hidden_dim`, `fusion_dim`,
`head_hidden`, dropout, embedding dims, `epoch_cap=150`) as `07g` --
this branch changes exactly ONE thing: every `GATv2Conv` inside
`SVGEncoder`/`TVGEncoder`/`UnifiedEncoder` is swapped for `GINEConv`
(`src/models.py`'s new `conv_type` switch, default `"gatv2"`, this run
uses `"gine"`). `configs/model_gin_comparison.yaml` is byte-identical to
`configs/model_capacity_revision.yaml` except for that one key, and this
notebook reuses `configs/eval_capacity_revision.yaml` **unchanged** (not
a copy) -- so encoder choice is the only variable between `07g` and
`07h`, not something that could drift via two slightly-different eval
YAMLs.

**GIN(E) vs GATv2, briefly:** GATv2 learns per-neighbor attention
weights when aggregating a node's neighbors. GIN(E) instead sums
neighbor messages (with edge features linearly folded in) through a
small MLP -- no learned per-neighbor weighting, but it's the most
WL-expressive standard message-passing scheme, and simpler (fewer
parameters per layer), which can matter on a dataset this small.
Trade-off: no attention-weight interpretability on this branch
(`GNNExplainer` still applies; attention-weight extraction doesn't).

**One real incompatibility, already fixed in `src/models.py`:**
`GINEConv` cannot run edge-attr-free the way `GATv2Conv` can --
scenario F's `same_location` edge (SVG `ego` <-> TVG `incident`, no real
edge features by design, see `unified_graph.py`) has no problem under
GATv2 (`edge_dim=None`, called with no `edge_attr` at all) but crashes
`GINEConv`, whose `message()` unconditionally indexes `edge_attr`. Fixed
by giving `same_location` a constant width-1 placeholder edge feature
ONLY on the `conv_type="gine"` path (same role `adjacent`'s existing
constant flag already plays elsewhere) -- the GATv2 path is completely
unchanged. Verified directly (SVG encoder, TVG encoder, unified/F
encoder including `same_location`, and a full `build_model()` forward
pass) before this notebook was written.

**Scenario G is skipped** -- it's the XGBoost tabular baseline, no GNN
encoder involved, so `conv_type` doesn't apply to it at all. `07g`'s `G`
result is the one to keep using for any G-inclusive comparison.

**Comparison cell at the end** loads `07g`'s own summary CSV alongside
this run's, merged on scenario, so the actual question -- does GIN(E)
beat GATv2 on this data -- has a direct answer instead of needing to be
read off two separate tables by eye.

No formal significance testing here, same caveat as every random-repeats
branch: descriptive aggregates (mean +/- std across 5 repeats) only.

GPU recommended.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched src/ files until pushed to GitHub.
# Skip this cell once the repo itself is updated -- needs the revised
# models.py (conv_type="gatv2"/"gine" switch + the same_location GINE
# fix) and train.py (head_hidden/head_dropout + per-epoch printing,
# from 07g), plus graph_datasets.py, unified_graph.py,
# baseline_features.py, evaluate.py, plot_history.py.
from google.colab import files
import shutil

print("Upload train.py, models.py, graph_datasets.py, unified_graph.py, "
      "baseline_features.py, evaluate.py, plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
# Reused UNCHANGED from 07g -- same eval scheme, only the model config differs.
with open(f"{REPO_DIR}/configs/eval_capacity_revision.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_gin_comparison.yaml") as f:
    model_cfg = yaml.safe_load(f)

# paths.yaml is per-city -- pooled outputs live under the "combined"
# section. NOT city_cfg["processed_dir"] anywhere below -- that field is
# an unresolved "${base_dir}/..." template string plain yaml.safe_load
# never substitutes; every path here is rebuilt from base_dir directly.
_bogor_base = Path(paths_cfg["per_city"]["bogor"]["base_dir"])
COMBINED_BASE_DIR = _bogor_base.parent / "combined"
COMBINED_PROCESSED_DIR = COMBINED_BASE_DIR / "processed"
if "combined" in paths_cfg:
    OUTPUTS_DIR = Path(paths_cfg["combined"]["outputs_dir"])
else:
    OUTPUTS_DIR = COMBINED_BASE_DIR / "outputs"
# separate checkpoint/metrics dirs from every other 07 branch, including 07g
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_gin_comparison"
METRICS_DIR = OUTPUTS_DIR / "metrics_gin_comparison"
for d in [CHECKPOINT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
# 07g's dirs, read-only here -- only used later by the comparison cell.
CAPACITY_REVISION_METRICS_DIR = OUTPUTS_DIR / "metrics_capacity_revision"

device = "cuda" if torch.cuda.is_available() else "cpu"
HEAD_DEPTH = model_cfg.get("head_depth", "mlp2")  # fixed, NOT swept -- see notebook intro
CONV_TYPE = model_cfg.get("conv_type", "gatv2")
config = {"batch_size": eval_cfg.get("batch_size") or 128,
          "epoch_cap": eval_cfg.get("epoch_cap") or 150,
          "warmup_epochs": eval_cfg.get("warmup_epochs") or 20,
          "patience": eval_cfg.get("patience") or 30,
          "lr_patience": eval_cfg.get("lr_patience") or 8,
          "lr": eval_cfg.get("lr") or 5e-3,
          "weight_decay": eval_cfg.get("weight_decay") or 1e-4,
          "fusion_dim": model_cfg.get("fusion_dim") or 128,
          "head_hidden": model_cfg.get("head_hidden") or 32,
          "head_dropout": model_cfg.get("head_dropout") or 0.35,
          "val_frac": eval_cfg.get("val_frac") or 0.15,
          "test_frac": eval_cfg.get("test_frac") or 0.15,
          "label_col": eval_cfg.get("label_col") or "label",
          "target_pos_frac": eval_cfg.get("target_pos_frac"),
          "threshold_method": eval_cfg.get("threshold_method") or "fixed",
          "threshold_fn_cost": eval_cfg.get("threshold_fn_cost") or 10.0,
          "threshold_fp_cost": eval_cfg.get("threshold_fp_cost") or 1.0,
          "num_workers": eval_cfg.get("num_workers") or 0,
          "use_amp": eval_cfg.get("use_amp", True)}
N_REPEATS = eval_cfg.get("n_repeats") or 5

train_frac = 1 - config["val_frac"] - config["test_frac"]
print("Device:", device, "| n_repeats:", N_REPEATS, "| head_depth:", HEAD_DEPTH, "(fixed, not swept)")
print("conv_type:", CONV_TYPE, "(the ONLY variable vs 07g)")
print("Split:", f"{train_frac:.0%}/{config['val_frac']:.0%}/{config['test_frac']:.0%}",
      "(train/val/test), stratified by", config["label_col"], "ONLY -- plain random, NOT by city")
print("Capacity (identical to 07g): hidden_dim=", model_cfg.get("hidden_dim"), " fusion_dim=", config["fusion_dim"],
      " head_hidden=", config["head_hidden"], " dropout=", model_cfg.get("dropout"),
      " head_dropout=", config["head_dropout"], sep="")
print("Epoch budget: warmup=", config["warmup_epochs"], " patience=", config["patience"],
      " epoch_cap=", config["epoch_cap"], sep="")

In [ ]:
import json
import pandas as pd
import graph_datasets as ds
import train as tr
import evaluate as ev
import models

INDEX_PATH = COMBINED_PROCESSED_DIR / "dataset_index.parquet"
index_df = pd.read_parquet(INDEX_PATH)
assert "city" in index_df.columns, (
    f"'{INDEX_PATH}' has no 'city' column -- this notebook needs 05b's "
    "pooled index, not a single city's dataset_index.parquet from 05.")
assert "uid" in index_df.columns, "expected 05b's city-prefixed 'uid' primary key column."

dataset = ds.PooledDualGraphDataset(index_df)
print(f"Dataset: {len(dataset)} points (pooled, GIN comparison branch)")
print(index_df.groupby("city")["label"].agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

_unified_cache_dir = _bogor_base / "interim" / "osm_cache"
with open(_unified_cache_dir / "highway_vocab.json") as f:
    HIGHWAY_VOCAB_SIZE = len(json.load(f))
with open(_unified_cache_dir / "building_type_vocab.json") as f:
    BUILDING_TYPE_VOCAB_SIZE = len(json.load(f))
print(f"Unified vocab (post-04b): highway={HIGHWAY_VOCAB_SIZE}, building_type={BUILDING_TYPE_VOCAB_SIZE}")

svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2,
                   cat_embed_dim=model_cfg.get("cat_embed_dim", 4), conv_type=CONV_TYPE)
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   building_type_vocab=BUILDING_TYPE_VOCAB_SIZE, highway_vocab=HIGHWAY_VOCAB_SIZE,
                   building_type_embed_dim=model_cfg.get("building_type_embed_dim", 16),
                   highway_embed_dim=model_cfg.get("highway_embed_dim", 8), conv_type=CONV_TYPE)
print("svg_kwargs:", svg_kwargs)
print("tvg_kwargs:", tvg_kwargs)

### Scenario A -- SVG only

In [ ]:
key = "A"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("A", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results = {key: results}
print(f"  {len(results)} repeat-runs complete.")

### Scenario B -- TVG only

In [ ]:
key = "B"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("B", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario C -- dual graph (concat)

In [ ]:
key = "C"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("C", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario D -- dual graph (late fusion)

In [ ]:
key = "D"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("D", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario E -- dual graph (cross-attention)

In [ ]:
key = "E"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("E", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario F -- unified merged graph

In [ ]:
key = "F"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("F", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Ablation B+ through F+

Same rationale as `07g`: one cell per scenario, split out for
independent run/monitor/interrupt.

#### Ablation B+

In [ ]:
key = "B_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("B", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation C+

In [ ]:
key = "C_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("C", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation D+

In [ ]:
key = "D_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("D", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation E+

In [ ]:
key = "E_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("E", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation F+

In [ ]:
key = "F_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}, {CONV_TYPE}) ===")
results = tr.run_scenario_random_repeats("F", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario G -- skipped on this branch

G is the XGBoost tabular baseline (`baseline_features.py`), no GNN
encoder involved at all -- `conv_type` has nothing to act on. `07g`'s
`G` result is the one to use for any comparison that needs to include
it; not re-run here.

## Aggregate + report every scenario

In [ ]:
agg_rows = []
for key, results in all_results.items():
    agg = ev.aggregate_fold_results(results)
    row = {"scenario": key}
    for metric, (mean, std) in agg.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(METRICS_DIR / "all_scenarios_summary_gin_comparison.csv", index=False)
display(agg_df)

## Threshold diagnostics

In [ ]:
threshold_rows = []
for key, results in all_results.items():
    method_counts = ev.summarize_categorical_field(results, "threshold_method")
    thresh_mean, thresh_std = ev.aggregate_fold_results(results).get("threshold_used", (float("nan"), float("nan")))
    threshold_rows.append({"scenario": key, "threshold_mean": thresh_mean, "threshold_std": thresh_std,
                            "methods_used": method_counts})

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(METRICS_DIR / "threshold_diagnostics_gin_comparison.csv", index=False)
display(threshold_df)

## Epoch-level diagnostics

Per-repeat training history saved under
`CHECKPOINT_DIR/{tag}_history/repeat{N}.json`, same JSON shape as every
other branch.

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = CHECKPOINT_DIR / f"A_{HEAD_DEPTH}_history" / "repeat0.json"
history = json.loads(history_path.read_text())

epochs = [h["epoch"] for h in history]
best_epoch = max(range(len(history)), key=lambda i: history[i]["val_pr_auc"])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs, [h["train_loss"] for h in history], label="train_loss", color="tab:blue")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train_loss", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(epochs, [h["val_pr_auc"] for h in history], label="val_pr_auc", color="tab:orange")
ax2.plot(epochs, [h["val_auroc"] for h in history], label="val_auroc", color="tab:green")
ax2.axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch ({best_epoch})")
ax2.set_ylabel("val metric")

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
plt.title(f"repeat0 history ({history_path.name}, conv_type={CONV_TYPE})")
plt.tight_layout()
plt.show()

## GIN(E) vs GATv2 -- direct comparison against 07g

Loads `07g`'s own summary CSV (same capacity numbers, same split, GATv2
encoders) alongside this run's, merged on `scenario`, so the actual
question this branch exists to answer -- does swapping the encoder move
PR-AUC/AUROC -- has a direct answer rather than needing two separate
tables read side by side.

In [ ]:
capacity_revision_summary_path = CAPACITY_REVISION_METRICS_DIR / "all_scenarios_summary_capacity_revision.csv"
if capacity_revision_summary_path.exists():
    gatv2_df = pd.read_csv(capacity_revision_summary_path)
    compare = agg_df[["scenario", "pr_auc_mean", "pr_auc_std", "auroc_mean", "auroc_std"]].merge(
        gatv2_df[["scenario", "pr_auc_mean", "pr_auc_std", "auroc_mean", "auroc_std"]],
        on="scenario", suffixes=("_gine", "_gatv2"))
    compare["pr_auc_delta_gine_minus_gatv2"] = compare["pr_auc_mean_gine"] - compare["pr_auc_mean_gatv2"]
    compare["auroc_delta_gine_minus_gatv2"] = compare["auroc_mean_gine"] - compare["auroc_mean_gatv2"]
    compare.to_csv(METRICS_DIR / "gine_vs_gatv2_comparison.csv", index=False)
    display(compare)
    n_gine_better = (compare["pr_auc_delta_gine_minus_gatv2"] > 0).sum()
    print(f"GINE has higher mean PR-AUC than GATv2 on {n_gine_better}/{len(compare)} scenarios.")
else:
    print("07g's summary CSV not found yet -- run 07g first to compare.")

In [ ]:
print("GIN(E) comparison branch complete.")
print(f"Every scenario A-F (+ ablations) trained with conv_type={CONV_TYPE},")
print("identical capacity/split/epoch settings to 07g (GATv2). See the comparison")
print("cell above (or metrics_gin_comparison/gine_vs_gatv2_comparison.csv) for")
print("whether the encoder swap actually moved PR-AUC/AUROC.")